# Build RAG index (`corpus.faiss` + `chunks.parquet` + `manifest.json`)

Run on a **cluster or serverless** notebook attached to your workspace. Prerequisites:

- Table `main.india_legal.legal_rag_corpus` populated (ingestion notebook).
- **Run the next code cell first** — it `pip install`s this repo in editable mode so `import nyaya_dhwani` works. Edit `REPO_ROOT` to match your clone (Workspace sidebar → right-click repo → **Copy path**).

If you still see `ModuleNotFoundError: nyaya_dhwani` after that, run **`dbutils.library.restartPython()`** once, then run all cells from the top.

Output default: `/Volumes/main/india_legal/legal_files/nyaya_index/`

See [docs/WORKSPACE_SETUP.md](../docs/WORKSPACE_SETUP.md) for secrets and Git.

In [ ]:
# --- Install package from your Repos checkout (edit REPO_ROOT) ---
# Common paths (use Copy path from the left sidebar):
#   /Workspace/Repos/<you@domain>/nyaya-dhwani-hackathon
#   /Workspace/Users/<you@domain>/nyaya-dhwani-hackathon
import subprocess
import sys

REPO_ROOT = "/Workspace/Users/<YOUR_EMAIL>/nyaya-dhwani-hackathon"  # noqa: E501 — edit me

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO_ROOT}[rag,rag_embed]",
    ]
)
print(f"✅ Installed nyaya-dhwani from {REPO_ROOT} (extras: rag, rag_embed)")
%restart_python

In [ ]:
CATALOG = "main"
SCHEMA = "india_legal"
TABLE = "legal_rag_corpus"
OUT_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/legal_files/nyaya_index"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

pdf = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").select(
    "chunk_id", "source", "doc_type", "title", "text"
).toPandas()

print(pdf.shape)
pdf.head(3)

In [ ]:
from nyaya_dhwani.embedder import SentenceEmbedder
from nyaya_dhwani.index_builder import save_rag_artifacts

texts = pdf["text"].fillna("").astype(str).tolist()
embedder = SentenceEmbedder(model_name=EMBED_MODEL, normalize=True)
emb = embedder.encode(texts)
print(emb.shape)

import os
os.makedirs(OUT_DIR, exist_ok=True)

manifest = save_rag_artifacts(
    OUT_DIR,
    embeddings=emb,
    chunks_df=pdf,
    embedding_model=EMBED_MODEL,
    catalog=CATALOG,
    schema=SCHEMA,
    source_table=TABLE,
)
print(manifest.to_json())

In [ ]:
# Smoke test load
from nyaya_dhwani.retrieval import CorpusIndex

ci = CorpusIndex.load(OUT_DIR)
q = embedder.encode(["What is theft under BNS?"])
display(ci.search(q, k=5))